# Simplified Baseline Experiments - Fast & Interpretable

**Objective**: Compare simple, fast baseline models for sentiment analysis
- Logistic Regression (classification)
- Logistic Regression + Weighted Average (regression variant)
- Linear Regression + Nearest Label Assignment (regression variant)
- Naive Bayes (fast baseline)
- Linear SVC

**Evaluation Metric**: `1 - MAE/4` (competition metric)

**Key Focus**: 
- Use pre-detected languages stored in dataset
- Provide deeper analysis per model
- Fast execution on macOS
- Insights for future BERT/augmentation strategies

## Section 1: Import Required Libraries

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')
import json

# Add baselines module to path
sys.path.insert(0, str(Path.cwd()))

from src.preprocessing import PreprocessingFactory, LanguageDetector
from src.models import (
    BOWLogisticRegression, BOWLogisticRegressionWeighted, 
    BOWLinearRegression, BOWNaiveBayes, BOWLinearSVC,
    create_model_from_config
)
from src.utils import MetricsComputer, ExperimentLogger, DataAnalyzer
from src.experiments import ExperimentRunner, BaselineComparison

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

print("✓ All libraries imported successfully")

## Section 2: Load and Prepare Data

In [ ]:
# Load data with pre-detected languages
data_dir = Path('../data')

# Try to load data with language information
try:
    train_full = pd.read_csv(data_dir / 'train_w_language.csv')
    test_full = pd.read_csv(data_dir / 'test_w_language.csv')
    print("✓ Loaded data with pre-detected languages")
except FileNotFoundError:
    print("⚠ Language data not found, loading base data")
    train_full = pd.read_csv(data_dir / 'train.csv')
    test_full = pd.read_csv(data_dir / 'test.csv')
    
    # Detect languages
    print("Detecting languages...")
    detector = LanguageDetector()
    train_full['language'] = LanguageDetector.detect_language_batch(train_full)
    test_full['language'] = LanguageDetector.detect_language_batch(test_full)

print(f"\nTrain set size: {len(train_full)}")
print(f"Test set size: {len(test_full)}")
print(f"\nTrain label distribution:\n{train_full['label'].value_counts().sort_index()}")
print(f"\nTrain language distribution:\n{train_full['language'].value_counts()}")
print(f"\nFirst 3 samples:")
print(train_full.head(3))

In [ ]:
# Create train/validation split (90/10) with stratification
train_df, val_df = train_test_split(
    train_full, 
    test_size=0.1, 
    stratify=train_full['label'], 
    random_state=42
)

print(f"Train: {len(train_df)}, Validation: {len(val_df)}")
print(f"Train label distribution:\n{train_df['label'].value_counts().sort_index()}")

# Create output directory for results
output_dir = Path('outputs_v2')
output_dir.mkdir(exist_ok=True)
print(f"\n✓ Output directory: {output_dir}")

## Section 3: Compare Baseline Model Types

In [ ]:
# Run comprehensive baseline comparison
comparison = BaselineComparison(
    train_df=train_df,
    val_df=val_df,
    test_df=test_full,
    text_col='sentence',
    label_col='label',
    output_dir=str(output_dir / 'model_types')
)

model_results = comparison.compare_model_types()

print("\n" + "="*70)
print("MODEL TYPE COMPARISON RESULTS")
print("="*70)

# Create results summary
model_summary = []
for result in model_results:
    config_name = result['config']['name']
    mae = result['val_metrics']['mae']
    score = result['val_metrics']['competition_score']
    accuracy = result['val_metrics'].get('accuracy', 'N/A')
    model_summary.append({
        'Model': config_name,
        'Val MAE': f"{mae:.4f}",
        'Metric (1-MAE/4)': f"{score:.4f}",
        'Accuracy': f"{accuracy:.4f}" if accuracy != 'N/A' else accuracy,
    })

model_df = pd.DataFrame(model_summary)
print(model_df.to_string(index=False))

# Find best model
best_idx = np.argmin([r['val_metrics']['mae'] for r in model_results])
best_model = model_results[best_idx]
best_name = best_model['config']['name']
print(f"\n→ Best Model: {best_name} (MAE: {best_model['val_metrics']['mae']:.4f})")

## Section 4: Deep Analysis of Best Model

In [ ]:
# Extract predictions from best model
best_model = model_results[best_idx]
y_val_true = val_df['label'].values
y_val_pred = best_model['val_preds']

# Compute detailed error statistics
error_stats = DataAnalyzer.compute_error_statistics(y_val_true, y_val_pred)

print("\n" + "="*70)
print(f"DETAILED ANALYSIS: {best_name}")
print("="*70)
print(f"\nError Statistics:")
print(f"  Mean Error: {error_stats['mean_error']:.4f}")
print(f"  Median Error: {error_stats['median_error']:.4f}")
print(f"  Std Error: {error_stats['std_error']:.4f}")
print(f"  Max Error: {error_stats['max_error']}")
print(f"  Perfect Predictions: {error_stats['perfect_predictions']} ({100*error_stats['perfect_predictions']/len(y_val_true):.1f}%)")
print(f"  Large Errors (>=2): {error_stats['large_errors']} ({100*error_stats['large_errors']/len(y_val_true):.1f}%)")

print(f"\nError Distribution:")
for error_key, error_data in error_stats['error_distribution'].items():
    print(f"  {error_key}: {error_data['count']:4d} ({error_data['percentage']:5.1f}%)")

# Per-label metrics
label_metrics = DataAnalyzer.compute_label_distribution_metrics(y_val_true, y_val_pred)
print(f"\nPer-Label Performance:")
for label_key, metrics in label_metrics.items():
    print(f"  {label_key}: MAE={metrics['mae']:.4f}, Accuracy={metrics['accuracy']:.4f}, Count={metrics['count']}")

# Language performance (if available)
if 'language' in val_df.columns:
    lang_metrics = DataAnalyzer.analyze_language_performance(val_df, y_val_true, y_val_pred)
    print(f"\nPer-Language Performance:")
    for lang_key, metrics in lang_metrics.items():
        print(f"  {lang_key}: MAE={metrics['mae']:.4f}, Accuracy={metrics['accuracy']:.4f}, Count={metrics['count']}")

In [ ]:
# Visualize predictions and errors
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Error distribution histogram
errors = np.abs(y_val_true - y_val_pred)
axes[0, 0].hist(errors, bins=range(0, int(errors.max())+2), edgecolor='black', alpha=0.7, color='steelblue')
axes[0, 0].set_xlabel('Absolute Error')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title(f'Error Distribution - {best_name}')
axes[0, 0].axvline(errors.mean(), color='r', linestyle='--', linewidth=2, label=f'Mean: {errors.mean():.2f}')
axes[0, 0].legend()

# Plot 2: Predictions vs Ground Truth
axes[0, 1].scatter(y_val_true, y_val_pred, alpha=0.4, s=20, color='darkblue')
axes[0, 1].plot([0.5, 5.5], [0.5, 5.5], 'r--', lw=2, label='Perfect Predictions')
axes[0, 1].set_xlabel('Ground Truth')
axes[0, 1].set_ylabel('Predictions')
axes[0, 1].set_title('Predictions vs Ground Truth')
axes[0, 1].set_xlim(0.5, 5.5)
axes[0, 1].set_ylim(0.5, 5.5)
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Label-wise MAE
labels = sorted(np.unique(y_val_true))
label_maes = []
for label in labels:
    mask = y_val_true == label
    if mask.sum() > 0:
        mae = np.abs(y_val_true[mask] - y_val_pred[mask]).mean()
        label_maes.append(mae)
    else:
        label_maes.append(0)

axes[1, 0].bar([str(int(l)) for l in labels], label_maes, color='coral', edgecolor='black')
axes[1, 0].set_xlabel('Label')
axes[1, 0].set_ylabel('Mean Absolute Error')
axes[1, 0].set_title('MAE by Label')
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Plot 4: Prediction counts by label
pred_counts = pd.Series(y_val_pred).value_counts().sort_index()
true_counts = pd.Series(y_val_true).value_counts().sort_index()

x = np.arange(len(labels))
width = 0.35
axes[1, 1].bar(x - width/2, [true_counts.get(l, 0) for l in labels], width, label='True', color='steelblue', edgecolor='black')
axes[1, 1].bar(x + width/2, [pred_counts.get(l, 0) for l in labels], width, label='Predicted', color='coral', edgecolor='black')
axes[1, 1].set_xlabel('Label')
axes[1, 1].set_ylabel('Count')
axes[1, 1].set_title('Label Distribution: True vs Predicted')
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels([str(int(l)) for l in labels])
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(output_dir / 'best_model_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print("✓ Saved: best_model_analysis.png")

## Section 5: Feature Extraction Comparison

In [ ]:
# Compare different feature extraction strategies
feature_comparison = BaselineComparison(
    train_df=train_df,
    val_df=val_df,
    test_df=test_full,
    text_col='sentence',
    label_col='label',
    output_dir=str(output_dir / 'feature_extraction')
)

feature_results = feature_comparison.compare_feature_extraction()

print("\n" + "="*70)
print("FEATURE EXTRACTION COMPARISON")
print("="*70)

feature_summary = []
for result in feature_results:
    config_name = result['config']['name']
    mae = result['val_metrics']['mae']
    score = result['val_metrics']['competition_score']
    feature_summary.append({
        'Feature Strategy': config_name,
        'Val MAE': f"{mae:.4f}",
        'Metric (1-MAE/4)': f"{score:.4f}",
    })

feature_df = pd.DataFrame(feature_summary)
print(feature_df.to_string(index=False))

# Visualize
fig, ax = plt.subplots(figsize=(12, 5))
maes = [r['val_metrics']['mae'] for r in feature_results]
names = [r['config']['name'] for r in feature_results]
colors = ['green' if 'tfidf' in n else 'steelblue' for n in names]

ax.barh(names, maes, color=colors, edgecolor='black', alpha=0.7)
ax.set_xlabel('Validation MAE')
ax.set_title('Feature Extraction Strategy Comparison')
ax.invert_yaxis()
for i, (name, mae) in enumerate(zip(names, maes)):
    ax.text(mae, i, f' {mae:.4f}', va='center')

plt.tight_layout()
plt.savefig(output_dir / 'feature_extraction_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print("✓ Saved: feature_extraction_comparison.png")

## Section 6: Preprocessing Impact

In [ ]:
# Compare preprocessing strategies
prep_comparison = BaselineComparison(
    train_df=train_df,
    val_df=val_df,
    test_df=test_full,
    text_col='sentence',
    label_col='label',
    output_dir=str(output_dir / 'preprocessing')
)

prep_results = prep_comparison.compare_preprocessing()

print("\n" + "="*70)
print("PREPROCESSING IMPACT ANALYSIS")
print("="*70)

prep_summary = []
for result in prep_results:
    config_name = result['config']['preprocessing']
    mae = result['val_metrics']['mae']
    score = result['val_metrics']['competition_score']
    prep_summary.append({
        'Preprocessing': config_name,
        'Val MAE': f"{mae:.4f}",
        'Metric (1-MAE/4)': f"{score:.4f}",
    })

prep_df = pd.DataFrame(prep_summary)
print(prep_df.to_string(index=False))

# Visualize
fig, ax = plt.subplots(figsize=(10, 5))
maes = [r['val_metrics']['mae'] for r in prep_results]
names = [r['config']['preprocessing'] for r in prep_results]
colors = ['green' if 'aggressive' in n or 'extended' in n else 'steelblue' for n in names]

ax.barh(names, maes, color=colors, edgecolor='black', alpha=0.7)
ax.set_xlabel('Validation MAE')
ax.set_title('Preprocessing Strategy Comparison')
ax.invert_yaxis()
for i, (name, mae) in enumerate(zip(names, maes)):
    ax.text(mae, i, f' {mae:.4f}', va='center')

plt.tight_layout()
plt.savefig(output_dir / 'preprocessing_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print("✓ Saved: preprocessing_comparison.png")

## Section 7: Language-Specific vs Multilingual Models

In [ ]:
# Compare language-specific vs multilingual models
lang_comparison = BaselineComparison(
    train_df=train_df,
    val_df=val_df,
    test_df=test_full,
    text_col='sentence',
    label_col='label',
    output_dir=str(output_dir / 'language_comparison')
)

lang_results = lang_comparison.compare_language_specific()

print("\n" + "="*70)
print("LANGUAGE-SPECIFIC VS MULTILINGUAL COMPARISON")
print("="*70)

lang_summary = []
for lang_key, result in lang_results.items():
    mae = result['val_metrics']['mae']
    score = result['val_metrics']['competition_score']
    count = len(result.get('val_preds', []))
    lang_summary.append({
        'Approach': lang_key,
        'Val MAE': f"{mae:.4f}",
        'Metric (1-MAE/4)': f"{score:.4f}",
    })

lang_df = pd.DataFrame(lang_summary)
print(lang_df.to_string(index=False))

# Visualize
fig, ax = plt.subplots(figsize=(10, 5))
maes = [r['val_metrics']['mae'] for r in lang_results.values()]
names = list(lang_results.keys())
colors = ['green' if 'multilingual' in n else 'coral' for n in names]

ax.barh(names, maes, color=colors, edgecolor='black', alpha=0.7)
ax.set_xlabel('Validation MAE')
ax.set_title('Language-Specific vs Multilingual Approach')
ax.invert_yaxis()
for i, (name, mae) in enumerate(zip(names, maes)):
    ax.text(mae, i, f' {mae:.4f}', va='center')

plt.tight_layout()
plt.savefig(output_dir / 'language_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print("✓ Saved: language_comparison.png")

## Section 8: Comprehensive Results Summary & Insights

## Section 9: L1 Regularization Impact (Sparsity Enforcement)

In [ ]:
# Compare L1 vs L2 regularization (sparsity enforcement)
l1_comparison = BaselineComparison(
    train_df=train_df,
    val_df=val_df,
    test_df=test_full,
    text_col='sentence',
    label_col='label',
    output_dir=str(output_dir / 'l1_regularization')
)

l1_results = l1_comparison.compare_l1_regularization()

print("\n" + "="*70)
print("L1 REGULARIZATION IMPACT (Sparsity Analysis)")
print("="*70)

l1_summary = []
for result in l1_results:
    config_name = result['config']['name']
    mae = result['val_metrics']['mae']
    score = result['val_metrics']['competition_score']
    model_obj = result['model']
    sparsity = model_obj.get_sparsity() if hasattr(model_obj, 'get_sparsity') else 0
    
    l1_summary.append({
        'Configuration': config_name,
        'Val MAE': f"{mae:.4f}",
        'Metric (1-MAE/4)': f"{score:.4f}",
        'Sparsity %': f"{sparsity:.1f}",
    })

l1_df = pd.DataFrame(l1_summary)
print(l1_df.to_string(index=False))

# Visualize L1 results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

configs = [r['config']['name'] for r in l1_results]
maes = [r['val_metrics']['mae'] for r in l1_results]
sparsities = [r['model'].get_sparsity() if hasattr(r['model'], 'get_sparsity') else 0 for r in l1_results]

# Plot 1: MAE comparison
axes[0].bar(range(len(configs)), maes, color=['steelblue' if 'l2' in c else 'coral' for c in configs], 
            edgecolor='black', alpha=0.7)
axes[0].set_xticks(range(len(configs)))
axes[0].set_xticklabels([c.replace('regularization_', '') for c in configs], rotation=45, ha='right')
axes[0].set_ylabel('Validation MAE')
axes[0].set_title('L1 vs L2 Regularization - MAE')
axes[0].grid(True, alpha=0.3, axis='y')

# Plot 2: Sparsity
axes[1].bar(range(len(configs)), sparsities, color=['steelblue' if 'l2' in c else 'coral' for c in configs],
            edgecolor='black', alpha=0.7)
axes[1].set_xticks(range(len(configs)))
axes[1].set_xticklabels([c.replace('regularization_', '') for c in configs], rotation=45, ha='right')
axes[1].set_ylabel('Sparsity (%)')
axes[1].set_title('L1 vs L2 Regularization - Sparsity')
axes[1].set_ylim([0, 100])
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(output_dir / 'l1_regularization_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print("✓ Saved: l1_regularization_comparison.png")

## Section 10: Language-Aware Lemmatization

In [ ]:
# Compare lemmatization impact (English & German support)
lemma_comparison = BaselineComparison(
    train_df=train_df,
    val_df=val_df,
    test_df=test_full,
    text_col='sentence',
    label_col='label',
    output_dir=str(output_dir / 'lemmatization')
)

lemma_results = lemma_comparison.compare_lemmatization()

print("\n" + "="*70)
print("LANGUAGE-AWARE LEMMATIZATION IMPACT")
print("="*70)

lemma_summary = []
for result in lemma_results:
    config_name = result['config']['name']
    mae = result['val_metrics']['mae']
    score = result['val_metrics']['competition_score']
    model_obj = result['model']
    sparsity = model_obj.get_sparsity() if hasattr(model_obj, 'get_sparsity') else None
    
    lemma_summary.append({
        'Configuration': config_name,
        'Val MAE': f"{mae:.4f}",
        'Metric (1-MAE/4)': f"{score:.4f}",
        'Sparsity %': f"{sparsity:.1f}" if sparsity else "N/A",
    })

lemma_df = pd.DataFrame(lemma_summary)
print(lemma_df.to_string(index=False))

# Visualize lemmatization results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

configs = [r['config']['name'] for r in lemma_results]
maes = [r['val_metrics']['mae'] for r in lemma_results]
colors = ['steelblue' if 'enabled' in c or 'weighted' in c else 'coral' for c in configs]

# Plot 1: MAE comparison
axes[0].bar(range(len(configs)), maes, color=colors, edgecolor='black', alpha=0.7)
axes[0].set_xticks(range(len(configs)))
axes[0].set_xticklabels([c.replace('lemmatization_', '') for c in configs], rotation=45, ha='right')
axes[0].set_ylabel('Validation MAE')
axes[0].set_title('Lemmatization Impact - MAE Comparison')
axes[0].grid(True, alpha=0.3, axis='y')
for i, mae in enumerate(maes):
    axes[0].text(i, mae + 0.005, f'{mae:.4f}', ha='center', va='bottom', fontsize=9)

# Plot 2: Improvement over baseline
baseline_mae = maes[0]  # lemmatization_none
improvements = [(baseline_mae - mae) / baseline_mae * 100 for mae in maes]
colors_improvement = ['gray' if 'none' in c else ('green' if imp > 0 else 'red') for c, imp in zip(configs, improvements)]

axes[1].bar(range(len(configs)), improvements, color=colors_improvement, edgecolor='black', alpha=0.7)
axes[1].set_xticks(range(len(configs)))
axes[1].set_xticklabels([c.replace('lemmatization_', '') for c in configs], rotation=45, ha='right')
axes[1].set_ylabel('MAE Improvement (%)')
axes[1].set_title('Lemmatization Impact - % Improvement over Baseline')
axes[1].axhline(0, color='black', linestyle='-', linewidth=0.8)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(output_dir / 'lemmatization_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print("✓ Saved: lemmatization_comparison.png")

In [ ]:
# Final summary with all 6 comparisons
print("\n" + "="*70)
print("FINAL SUMMARY - ALL BASELINE COMPARISONS")
print("="*70)

summary_data = [
    ('Model Types', np.min([r['val_metrics']['mae'] for r in model_results])),
    ('Feature Extraction', np.min([r['val_metrics']['mae'] for r in feature_results])),
    ('Preprocessing', np.min([r['val_metrics']['mae'] for r in prep_results])),
    ('Language Approach', np.min([r['val_metrics']['mae'] for r in lang_results.values()])),
    ('Lemmatization', np.min([r['val_metrics']['mae'] for r in lemma_results])),
]

summary_df = pd.DataFrame(summary_data, columns=['Comparison', 'Best MAE'])
summary_df['Score (1-MAE/4)'] = (1 - summary_df['Best MAE'] / 4).round(4)
summary_df = summary_df.sort_values('Best MAE')

print(summary_df.to_string(index=False))

print(f"\n📊 Overall Best Performer:")
print(f"   Approach: {summary_df.iloc[0]['Comparison']}")
print(f"   MAE: {summary_df.iloc[0]['Best MAE']:.4f}")
print(f"   Score: {summary_df.iloc[0]['Score (1-MAE/4)']:.4f}")

print(f"\n✅ All baseline comparisons complete!")
print(f"📂 Results saved to: {output_dir}/")
print(f"   - model_types/")
print(f"   - feature_extraction/")
print(f"   - preprocessing/")
print(f"   - language_comparison/")
print(f"   - l1_regularization/")
print(f"   - lemmatization/")

print(f"\n💡 Next Steps:")
print(f"   1. Review lemmatization vs baseline (language-aware grouping)")
print(f"   2. Check L1 sparsity for interpretability")
print(f"   3. Use best configuration for BERT fine-tuning")
print(f"   4. Analyze high-error cases for data augmentation")

In [ ]:
# Save comprehensive summary
print("\n" + "="*70)
print("COMPREHENSIVE BASELINE SUMMARY")
print("="*70)

# Consolidate all results
all_results_data = []

# Model types
for result in model_results:
    all_results_data.append({
        'Category': 'Model Type',
        'Approach': result['config']['name'],
        'Val MAE': result['val_metrics']['mae'],
        'Metric (1-MAE/4)': result['val_metrics']['competition_score'],
    })

# Feature extraction
for result in feature_results:
    all_results_data.append({
        'Category': 'Feature Extraction',
        'Approach': result['config']['name'],
        'Val MAE': result['val_metrics']['mae'],
        'Metric (1-MAE/4)': result['val_metrics']['competition_score'],
    })

# Preprocessing
for result in prep_results:
    all_results_data.append({
        'Category': 'Preprocessing',
        'Approach': f"prep_{result['config']['preprocessing']}",
        'Val MAE': result['val_metrics']['mae'],
        'Metric (1-MAE/4)': result['val_metrics']['competition_score'],
    })

# Language
for lang_key, result in lang_results.items():
    all_results_data.append({
        'Category': 'Language Approach',
        'Approach': lang_key,
        'Val MAE': result['val_metrics']['mae'],
        'Metric (1-MAE/4)': result['val_metrics']['competition_score'],
    })

all_results_df = pd.DataFrame(all_results_data)
all_results_df = all_results_df.sort_values('Val MAE')

print("\nTop 10 Results by MAE:")
print(all_results_df.head(10).to_string(index=False))

# Save to CSV
all_results_df.to_csv(output_dir / 'all_baseline_results.csv', index=False)
print(f"\n✓ Saved comprehensive results to: all_baseline_results.csv")

In [ ]:
# Generate summary insights
summary_insights = f"""
╔════════════════════════════════════════════════════════════════════════════╗
║           SENTIMENT ANALYSIS BASELINE SUMMARY & INSIGHTS                   ║
╚════════════════════════════════════════════════════════════════════════════╝

1. EVALUATION METRIC
   └─ Score = 1 - MAE/4 (higher is better)
   └─ Range: 0.0 to 1.0 (0 = all predictions off by 4, 1 = perfect)

2. BASELINE MODEL PERFORMANCE
   Best Model: {best_name}
   └─ Validation MAE: {best_model['val_metrics']['mae']:.4f}
   └─ Competition Score: {best_model['val_metrics']['competition_score']:.4f}
   └─ Perfect Predictions: {error_stats['perfect_predictions']} ({100*error_stats['perfect_predictions']/len(y_val_true):.1f}%)
   └─ Large Errors (≥2): {error_stats['large_errors']} ({100*error_stats['large_errors']/len(y_val_true):.1f}%)

3. ERROR ANALYSIS
   Mean Error:    {error_stats['mean_error']:.4f}
   Median Error:  {error_stats['median_error']:.4f}
   Std Error:     {error_stats['std_error']:.4f}
   Max Error:     {error_stats['max_error']}

4. BEST STRATEGIES
   Feature Extraction: {feature_df.iloc[0]['Feature Strategy']} (MAE: {float(feature_df.iloc[0]['Val MAE']):.4f})
   Preprocessing:     {prep_df.iloc[0]['Preprocessing']} (MAE: {float(prep_df.iloc[0]['Val MAE']):.4f})
   Language Approach: {list(lang_results.keys())[np.argmin([r['val_metrics']['mae'] for r in lang_results.values()])]}

5. KEY FINDINGS
   • Simple logistic regression is fast and interpretable baseline
   • Feature engineering (TF-IDF vs Count, n-grams) has notable impact
   • Preprocessing has moderate impact on performance
   • Language-specific models may offer advantages if data is imbalanced

6. INSIGHTS FOR FUTURE IMPROVEMENTS (BERT + Data Augmentation)
   • Large errors ({error_stats['large_errors']} cases) suggest context/nuance is important
   • Performance varies by label → label-specific augmentation could help
   • Language differences → multilingual BERT could be valuable
   • Error patterns suggest room for sequence-based models

7. RECOMMENDATIONS
   ✓ Use '{best_name}' as production baseline
   ✓ Consider TF-IDF with bigrams for feature extraction
   ✓ Investigate high-error samples for augmentation targets
   ✓ Next step: Try BERT-based models with targeted augmentation

8. FILES SAVED TO: {output_dir}
   • all_baseline_results.csv - Complete results table
   • best_model_analysis.png - Best model visualizations
   • feature_extraction_comparison.png
   • preprocessing_comparison.png
   • language_comparison.png
"""

print(summary_insights)

# Save summary to file
with open(output_dir / 'SUMMARY.txt', 'w') as f:
    f.write(summary_insights)
print(f"✓ Summary saved to: SUMMARY.txt")

In [ ]:
# Analyze failure cases for insights
print("\n" + "="*70)
print("FAILURE CASE ANALYSIS (for data augmentation insights)")
print("="*70)

# Get high-error samples
high_error_df = DataAnalyzer.analyze_failures(val_df.copy(), y_val_true, y_val_pred, threshold=2)
print(f"\nTotal high-error cases: {len(high_error_df)}")
print(f"\nTop 10 High-Error Examples:")
print("-" * 70)

for idx, (i, row) in enumerate(high_error_df.head(10).iterrows()):
    print(f"\n{idx+1}. Error: {int(row['error'])} | True: {int(row['true_label'])} → Predicted: {int(row['pred_label'])}")
    if 'language' in row:
        print(f"   Language: {row['language']}")
    text_preview = row['sentence'][:100].replace('\n', ' ')
    print(f"   Text: {text_preview}..." if len(row['sentence']) > 100 else f"   Text: {text_preview}")
    if 'text_length' in row:
        print(f"   Length: {int(row['text_length'])} chars, {int(row['text_tokens'])} tokens")

# Summary statistics for high-error cases
print(f"\n\nHigh-Error Case Statistics:")
print(f"  Average text length: {high_error_df['text_length'].mean():.0f} chars")
print(f"  Average text length: {high_error_df['text_tokens'].mean():.0f} tokens")
if 'language' in high_error_df.columns:
    print(f"  Language distribution:")
    for lang, count in high_error_df['language'].value_counts().items():
        print(f"    {lang}: {count} ({100*count/len(high_error_df):.1f}%)")

# Save high-error cases for review
high_error_df.to_csv(output_dir / 'high_error_cases.csv', index=False)
print(f"\n✓ High-error cases saved to: high_error_cases.csv (useful for data augmentation)")

In [ ]:
# Final comparison visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Plot 1: Model types comparison
model_names = [r['config']['name'] for r in model_results]
model_maes = [r['val_metrics']['mae'] for r in model_results]
axes[0, 0].bar(range(len(model_names)), model_maes, color='steelblue', edgecolor='black', alpha=0.7)
axes[0, 0].set_xticks(range(len(model_names)))
axes[0, 0].set_xticklabels([n.replace('baseline_', '') for n in model_names], rotation=45, ha='right')
axes[0, 0].set_ylabel('Validation MAE')
axes[0, 0].set_title('Model Type Comparison')
axes[0, 0].grid(True, alpha=0.3, axis='y')
axes[0, 0].axhline(np.min(model_maes), color='r', linestyle='--', alpha=0.5, label='Best')

# Plot 2: Feature extraction comparison
feature_names = [r['config']['name'].replace('features_', '') for r in feature_results]
feature_maes = [r['val_metrics']['mae'] for r in feature_results]
axes[0, 1].bar(range(len(feature_names)), feature_maes, color='coral', edgecolor='black', alpha=0.7)
axes[0, 1].set_xticks(range(len(feature_names)))
axes[0, 1].set_xticklabels(feature_names, rotation=45, ha='right')
axes[0, 1].set_ylabel('Validation MAE')
axes[0, 1].set_title('Feature Extraction Strategy Comparison')
axes[0, 1].grid(True, alpha=0.3, axis='y')
axes[0, 1].axhline(np.min(feature_maes), color='r', linestyle='--', alpha=0.5, label='Best')

# Plot 3: Preprocessing comparison
prep_names = [r['config']['preprocessing'] for r in prep_results]
prep_maes = [r['val_metrics']['mae'] for r in prep_results]
axes[1, 0].bar(range(len(prep_names)), prep_maes, color='lightgreen', edgecolor='black', alpha=0.7)
axes[1, 0].set_xticks(range(len(prep_names)))
axes[1, 0].set_xticklabels(prep_names, rotation=45, ha='right')
axes[1, 0].set_ylabel('Validation MAE')
axes[1, 0].set_title('Preprocessing Strategy Comparison')
axes[1, 0].grid(True, alpha=0.3, axis='y')
axes[1, 0].axhline(np.min(prep_maes), color='r', linestyle='--', alpha=0.5, label='Best')

# Plot 4: Category-wise best results
categories = ['Model Type', 'Feature', 'Preprocessing', 'Language']
best_per_category = [
    np.min([r['val_metrics']['mae'] for r in model_results]),
    np.min([r['val_metrics']['mae'] for r in feature_results]),
    np.min([r['val_metrics']['mae'] for r in prep_results]),
    np.min([r['val_metrics']['mae'] for r in lang_results.values()]),
]
scores = [1 - mae/4 for mae in best_per_category]
axes[1, 1].bar(categories, scores, color=['steelblue', 'coral', 'lightgreen', 'plum'], edgecolor='black', alpha=0.7)
axes[1, 1].set_ylabel('Best Competition Score (1-MAE/4)')
axes[1, 1].set_title('Best Score by Category')
axes[1, 1].set_ylim([0, 1])
axes[1, 1].grid(True, alpha=0.3, axis='y')
for i, score in enumerate(scores):
    axes[1, 1].text(i, score + 0.02, f'{score:.4f}', ha='center')

plt.tight_layout()
plt.savefig(output_dir / 'comprehensive_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print("✓ Saved: comprehensive_comparison.png")